In [ ]:
"""
Exercise 1 - Perceptron Decision Boundary and Linear Separability Challenge
Dataset : Banknote Authentication (UCI id=267)

Place either of these next to this script (or pass the path as argv[1]):
    BankNote_Authentication.csv        (Kaggle version, has a header row)
    data_banknote_authentication.txt   (UCI version, no header)

If neither file is found, a statistically similar synthetic dataset is
generated so the program still runs end-to-end.

Columns: variance, skewness, curtosis, entropy  ->  class (0 = genuine, 1 = forged)
"""

import os
import sys
import itertools
import random

import numpy as np

# --------------------------------------------------------------------------- #
# Configuration
# --------------------------------------------------------------------------- #
FEATURE_NAMES = ["Variance", "Skewness", "Curtosis", "Entropy"]
CLASS_NAMES = {0: "Genuine", 1: "Forged"}
LEARNING_RATE = 0.01
MAX_EPOCHS = 100
TEST_RATIO = 0.30
SEED = 42

CANDIDATE_FILES = [
    "BankNote_Authentication.csv",
    "bank_note_authentication.csv",
    "data_banknote_authentication.txt",
    "data_banknote_authentication.csv",
    "/mnt/user-data/uploads/BankNote_Authentication.csv",
    "/mnt/user-data/uploads/data_banknote_authentication.txt",
]


# --------------------------------------------------------------------------- #
# 1. Data handling
# --------------------------------------------------------------------------- #
def _synthetic_dataset(n=1372, seed=SEED):
    """Fallback data with class statistics close to the real banknote set."""
    rng = np.random.default_rng(seed)
    n0, n1 = 762, n - 762                       # real split is 762 / 610
    genuine = rng.multivariate_normal(
        mean=[2.80, 5.60, 0.30, -1.15],
        cov=np.diag([2.6, 14.0, 10.0, 3.5]),
        size=n0,
    )
    forged = rng.multivariate_normal(
        mean=[-2.40, -1.60, 2.90, -1.25],
        cov=np.diag([2.4, 16.0, 13.0, 3.2]),
        size=n1,
    )
    X = np.vstack([genuine, forged])
    y = np.hstack([np.zeros(n0, dtype=int), np.ones(n1, dtype=int)])
    idx = rng.permutation(len(y))
    return X[idx], y[idx]


def load_dataset(path=None):
    """Return (X, y, source_description)."""
    paths = [path] if path else CANDIDATE_FILES
    for p in paths:
        if p and os.path.exists(p):
            with open(p) as fh:
                first = fh.readline()
            skip = 1 if any(c.isalpha() for c in first) else 0
            data = np.loadtxt(p, delimiter=",", skiprows=skip)
            return data[:, :4], data[:, 4].astype(int), os.path.basename(p)
    return (*_synthetic_dataset(), "SYNTHETIC (dataset file not found)")


def normalize(X):
    """Min-max normalisation of every column to [0, 1]."""
    lo, hi = X.min(axis=0), X.max(axis=0)
    span = np.where(hi - lo == 0, 1.0, hi - lo)
    return (X - lo) / span


def stratified_split(X, y, test_ratio=TEST_RATIO, seed=SEED):
    rng = np.random.default_rng(seed)
    tr_idx, te_idx = [], []
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        cut = int(round(len(idx) * (1 - test_ratio)))
        tr_idx.extend(idx[:cut])
        te_idx.extend(idx[cut:])
    tr_idx, te_idx = np.array(tr_idx), np.array(te_idx)
    rng.shuffle(tr_idx)
    rng.shuffle(te_idx)
    return X[tr_idx], X[te_idx], y[tr_idx], y[te_idx]


# --------------------------------------------------------------------------- #
# 2. Single-Layer Perceptron
# --------------------------------------------------------------------------- #
class Perceptron:
    """Single-layer perceptron with a unit-step activation function."""

    def __init__(self, lr=LEARNING_RATE, max_epochs=MAX_EPOCHS, seed=SEED):
        self.lr = lr
        self.max_epochs = max_epochs
        self.seed = seed
        self.w = None
        self.b = 0.0
        self.epochs_run = 0
        self.converged = False
        self.errors_per_epoch = []

    @staticmethod
    def step(z):
        return np.where(z >= 0, 1, 0)

    def net_input(self, X):
        return np.dot(X, self.w) + self.b

    def predict(self, X):
        return self.step(self.net_input(X))

    def fit(self, X, y, order="original"):
        rng = np.random.default_rng(self.seed)
        n_samples, n_features = X.shape
        self.w = rng.normal(0.0, 0.01, n_features)
        self.b = 0.0
        self.errors_per_epoch = []
        self.converged = False

        idx = self._order_indices(y, order, rng)

        for epoch in range(1, self.max_epochs + 1):
            errors = 0
            if order == "random":                       # reshuffle each epoch
                idx = rng.permutation(n_samples)
            for i in idx:
                xi, target = X[i], y[i]
                update = self.lr * (target - self.step(np.dot(xi, self.w) + self.b))
                if update != 0.0:
                    self.w += update * xi
                    self.b += update
                    errors += 1
            self.errors_per_epoch.append(errors)
            self.epochs_run = epoch
            if errors == 0:
                self.converged = True
                break
        return self

    @staticmethod
    def _order_indices(y, order, rng):
        n = len(y)
        if order == "original":
            return np.arange(n)
        if order == "reverse":
            return np.arange(n)[::-1]
        if order == "random":
            return rng.permutation(n)
        if order == "classwise":
            return np.concatenate([np.where(y == c)[0] for c in np.unique(y)])
        raise ValueError(f"unknown order: {order}")


def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred) * 100.0)


# --------------------------------------------------------------------------- #
# 3. Presentation helpers
# --------------------------------------------------------------------------- #
def banner(title):
    print("\n" + "=" * 78)
    print(title.center(78))
    print("=" * 78)


def weight_string(w, b, names):
    parts = [f"{n}={wi:+.4f}" for n, wi in zip(names, w)]
    return "  ".join(parts) + f"   bias={b:+.4f}"


def pause():
    input("\nPress <Enter> to return to the menu...")


# --------------------------------------------------------------------------- #
# Option 1 - Banknote Classification using Perceptron
# --------------------------------------------------------------------------- #
def option_1(D):
    banner("1. BANKNOTE CLASSIFICATION USING PERCEPTRON")
    model = Perceptron().fit(D["Xtr"], D["ytr"])
    z = model.net_input(D["Xte"])
    pred = model.step(z)

    print(f"Features used   : {', '.join(FEATURE_NAMES)}")
    print(f"Training samples: {len(D['ytr'])}   Test samples: {len(D['yte'])}")
    print(f"Epochs run      : {model.epochs_run}  "
          f"(converged: {'YES' if model.converged else 'NO'})")
    print(f"Final weights   : {weight_string(model.w, model.b, FEATURE_NAMES)}")

    print("\n{:>5} {:>10} {:>11} {:>14} {:>9}".format(
        "S.No", "Actual", "Predicted", "Weighted Sum", "Result"))
    print("-" * 78)
    show = min(25, len(z))
    for i in range(show):
        print("{:>5} {:>10} {:>11} {:>14.5f} {:>9}".format(
            i + 1, CLASS_NAMES[D["yte"][i]], CLASS_NAMES[pred[i]],
            z[i], "OK" if pred[i] == D["yte"][i] else "WRONG"))
    if len(z) > show:
        print(f"... {len(z) - show} more test samples not shown ...")

    wrong = int(np.sum(pred != D["yte"]))
    print("-" * 78)
    print(f"Misclassified   : {wrong} / {len(D['yte'])}")
    print(f"Test accuracy   : {accuracy(D['yte'], pred):.2f} %")
    print(f"Train accuracy  : {accuracy(D['ytr'], model.predict(D['Xtr'])):.2f} %")
    if not model.converged:
        print("\nNote: zero training errors were never reached, so this data is not")
        print("perfectly linearly separable; the perceptron keeps adjusting the")
        print("hyperplane and the run stops at the epoch limit.")
    pause()


# --------------------------------------------------------------------------- #
# Option 2 - Decision Boundary Explorer
# --------------------------------------------------------------------------- #
def option_2(D):
    banner("2. DECISION BOUNDARY EXPLORER")
    for i, n in enumerate(FEATURE_NAMES):
        print(f"  {i + 1}. {n}")
    try:
        f1 = int(input("Select first feature  (1-4): ")) - 1
        f2 = int(input("Select second feature (1-4): ")) - 1
    except ValueError:
        print("Invalid input.")
        return pause()
    if not (0 <= f1 < 4 and 0 <= f2 < 4) or f1 == f2:
        print("Choose two different features between 1 and 4.")
        return pause()

    names = [FEATURE_NAMES[f1], FEATURE_NAMES[f2]]
    Xtr, Xte = D["Xtr"][:, [f1, f2]], D["Xte"][:, [f1, f2]]
    model = Perceptron().fit(Xtr, D["ytr"])
    z = model.net_input(Xte)
    pred = model.step(z)
    w1, w2, b = model.w[0], model.w[1], model.b

    print(f"\nFeature pair    : {names[0]} vs {names[1]}")
    print(f"Epochs run      : {model.epochs_run}  "
          f"(converged: {'YES' if model.converged else 'NO'})")
    print(f"Final weights   : w1={w1:+.5f}  w2={w2:+.5f}")
    print(f"Final bias      : b ={b:+.5f}")
    print(f"\nDecision boundary (implicit form):")
    print(f"    ({w1:+.5f})*{names[0]} + ({w2:+.5f})*{names[1]} + ({b:+.5f}) = 0")
    if abs(w2) > 1e-9:
        print(f"Slope-intercept form:")
        print(f"    {names[1]} = {-w1 / w2:+.5f} * {names[0]} {-b / w2:+.5f}")
    else:
        print(f"Vertical boundary : {names[0]} = {-b / w1:+.5f}")

    print("\n{:>5} {:>10} {:>10} {:>13} {:>12} {:>9}".format(
        "S.No", names[0][:10], names[1][:10], "Weighted Sum", "Side", "Predicted"))
    print("-" * 78)
    show = min(20, len(z))
    for i in range(show):
        side = "POSITIVE" if z[i] >= 0 else "NEGATIVE"
        print("{:>5} {:>10.4f} {:>10.4f} {:>13.5f} {:>12} {:>9}".format(
            i + 1, Xte[i, 0], Xte[i, 1], z[i], side, CLASS_NAMES[pred[i]]))
    if len(z) > show:
        print(f"... {len(z) - show} more test samples not shown ...")

    pos, neg = int(np.sum(z >= 0)), int(np.sum(z < 0))
    print("-" * 78)
    print(f"Positive side   : {pos} samples   Negative side: {neg} samples")
    print(f"Test accuracy   : {accuracy(D['yte'], pred):.2f} %")
    pause()


# --------------------------------------------------------------------------- #
# Option 3 - Feature Pair Separability Challenge
# --------------------------------------------------------------------------- #
def feature_pair_results(D):
    rows = []
    for f1, f2 in itertools.combinations(range(4), 2):
        m = Perceptron().fit(D["Xtr"][:, [f1, f2]], D["ytr"])
        pred = m.predict(D["Xte"][:, [f1, f2]])
        acc = accuracy(D["yte"], pred)
        rows.append({
            "pair": (f1, f2),
            "label": f"{FEATURE_NAMES[f1]} + {FEATURE_NAMES[f2]}",
            "epochs": m.epochs_run,
            "acc": acc,
            "wrong": int(np.sum(pred != D["yte"])),
            "converged": m.converged,
        })
    rows.sort(key=lambda r: (-r["acc"], r["epochs"]))
    return rows


def option_3(D):
    banner("3. FEATURE PAIR SEPARABILITY CHALLENGE")
    rows = feature_pair_results(D)
    print("{:>4} {:<28} {:>8} {:>10} {:>14} {:>11}".format(
        "Rank", "Feature Pair", "Epochs", "Accuracy", "Misclassified", "Converged"))
    print("-" * 78)
    for r, row in enumerate(rows, 1):
        print("{:>4} {:<28} {:>8} {:>9.2f}% {:>14} {:>11}".format(
            r, row["label"], row["epochs"], row["acc"], row["wrong"],
            "YES" if row["converged"] else "NO"))
    best = rows[0]
    print("-" * 78)
    print(f"MOST LINEARLY SEPARABLE PAIR : {best['label']}")
    print(f"   accuracy {best['acc']:.2f}% | {best['epochs']} epochs | "
          f"{best['wrong']} misclassified")
    print(f"LEAST SEPARABLE PAIR         : {rows[-1]['label']} "
          f"({rows[-1]['acc']:.2f}%)")
    pause()


# --------------------------------------------------------------------------- #
# Option 4 - Training Sample Order Challenge
# --------------------------------------------------------------------------- #
def order_results(D):
    rows = []
    for order, label in [("original", "Original"), ("reverse", "Reverse"),
                         ("random", "Random"), ("classwise", "Class-wise")]:
        m = Perceptron().fit(D["Xtr"], D["ytr"], order=order)
        pred = m.predict(D["Xte"])
        rows.append({
            "label": label, "order": order, "w": m.w.copy(), "b": m.b,
            "epochs": m.epochs_run, "acc": accuracy(D["yte"], pred),
            "converged": m.converged,
        })
    return rows


def option_4(D):
    banner("4. TRAINING SAMPLE ORDER CHALLENGE")
    rows = order_results(D)
    print("{:<12} {:>8} {:>10} {:>11}   {}".format(
        "Order", "Epochs", "Accuracy", "Converged", "Bias"))
    print("-" * 78)
    for r in rows:
        print("{:<12} {:>8} {:>9.2f}% {:>11}   {:+.5f}".format(
            r["label"], r["epochs"], r["acc"],
            "YES" if r["converged"] else "NO", r["b"]))

    print("\nFinal weight vectors")
    print("-" * 78)
    print("{:<12} {:>12} {:>12} {:>12} {:>12}".format("Order", *FEATURE_NAMES))
    for r in rows:
        print("{:<12} {:>12.5f} {:>12.5f} {:>12.5f} {:>12.5f}".format(
            r["label"], *r["w"]))

    ref = np.append(rows[0]["w"], rows[0]["b"])
    print("\nDeviation of each boundary from the 'Original' order boundary")
    print("-" * 78)
    different = False
    for r in rows[1:]:
        vec = np.append(r["w"], r["b"])
        dist = float(np.linalg.norm(vec - ref))
        cos = float(np.dot(vec, ref) /
                    (np.linalg.norm(vec) * np.linalg.norm(ref) + 1e-12))
        if dist > 1e-6:
            different = True
        print(f"{r['label']:<12} L2 difference = {dist:.5f}   "
              f"cosine similarity = {cos:.5f}")

    print("-" * 78)
    if different:
        print("CONCLUSION: sample order DOES change the final decision boundary.")
        print("The perceptron rule updates on each mistake, so a different visiting")
        print("order produces a different sequence of updates and a different (but")
        print("often equally valid) separating hyperplane.")
    else:
        print("CONCLUSION: all orders converged to the same boundary.")
    accs = [r["acc"] for r in rows]
    print(f"Accuracy spread: {min(accs):.2f}% - {max(accs):.2f}% "
          f"(range {max(accs) - min(accs):.2f} points)")
    pause()


# --------------------------------------------------------------------------- #
# Option 5 - Borderline Sample Detector
# --------------------------------------------------------------------------- #
BORDER_MARGIN = 0.05        # perpendicular distance in normalised feature space


def borderline_info(D, top_k=5, margin=BORDER_MARGIN):
    m = Perceptron().fit(D["Xtr"], D["ytr"])
    z = m.net_input(D["Xte"])
    pred = m.step(z)
    dist = np.abs(z) / (np.linalg.norm(m.w) + 1e-12)   # perpendicular distance
    order = np.argsort(dist)
    return m, z, pred, dist, order[:top_k], int(np.sum(dist < margin))


def option_5(D):
    banner("5. BORDERLINE SAMPLE DETECTOR")
    m, z, pred, dist, top, n_border = borderline_info(D)
    print("Threshold of the step function : weighted sum = 0")
    print(f"Final weights : {weight_string(m.w, m.b, FEATURE_NAMES)}")
    print(f"\nTop 5 samples closest to the decision boundary:")
    print("{:>5} {:>9} {:>9} {:>9} {:>9} {:>12} {:>10} {:>10}".format(
        "Test#", "Var", "Skew", "Curt", "Entr", "WeightedSum", "Distance", "Predicted"))
    print("-" * 78)
    for i in top:
        print("{:>5} {:>9.4f} {:>9.4f} {:>9.4f} {:>9.4f} {:>12.6f} {:>10.6f} {:>10}".format(
            int(i), *D["Xte"][i], z[i], dist[i], CLASS_NAMES[pred[i]]))
    print("-" * 78)
    for i in top:
        flag = "correct" if pred[i] == D["yte"][i] else "MISCLASSIFIED"
        print(f"  Test#{int(i):<4} actual={CLASS_NAMES[D['yte'][i]]:<8} "
              f"predicted={CLASS_NAMES[pred[i]]:<8} -> {flag}")
    print(f"\nBorderline samples (perpendicular distance < {BORDER_MARGIN}) : "
          f"{n_border} of {len(z)}")
    print("These samples sit in the uncertainty band; a small change in any")
    print("feature value would flip them to the other side of the hyperplane.")
    pause()


# --------------------------------------------------------------------------- #
# Option 6 - XOR Limitation Demonstrator
# --------------------------------------------------------------------------- #
LOGIC_GATES = {
    "AND":  np.array([0, 0, 0, 1]),
    "OR":   np.array([0, 1, 1, 1]),
    "NAND": np.array([1, 1, 1, 0]),
    "XOR":  np.array([0, 1, 1, 0]),
}
GATE_INPUTS = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)


def gate_results(max_epochs=MAX_EPOCHS):
    out = {}
    for gate, y in LOGIC_GATES.items():
        m = Perceptron(lr=0.1, max_epochs=max_epochs).fit(GATE_INPUTS, y)
        pred = m.predict(GATE_INPUTS)
        out[gate] = {
            "model": m, "y": y, "pred": pred,
            "acc": accuracy(y, pred), "converged": m.converged,
        }
    return out


def option_6(D):
    banner("6. XOR LIMITATION DEMONSTRATOR")
    res = gate_results()
    print(f"Maximum epochs allowed: {MAX_EPOCHS}\n")
    print("{:<7} {:>8} {:>11} {:>10} {:>26}".format(
        "Gate", "Epochs", "Converged", "Accuracy", "Verdict"))
    print("-" * 78)
    for gate in ["AND", "OR", "NAND", "XOR"]:
        r = res[gate]
        verdict = ("Linearly Separable" if r["converged"]
                   else "NOT Linearly Separable")
        print("{:<7} {:>8} {:>11} {:>9.1f}% {:>26}".format(
            gate, r["model"].epochs_run, "YES" if r["converged"] else "NO",
            r["acc"], verdict))

    print("\nTruth tables and perceptron output")
    print("-" * 78)
    print("{:>3} {:>3} | {}".format("x1", "x2",
          "  ".join(f"{g:<14}" for g in ["AND", "OR", "NAND", "XOR"])))
    for i, (x1, x2) in enumerate(GATE_INPUTS):
        cells = []
        for g in ["AND", "OR", "NAND", "XOR"]:
            t, p = res[g]["y"][i], res[g]["pred"][i]
            cells.append(f"t={t} p={p} {'ok' if t == p else 'X '}".ljust(14))
        print("{:>3.0f} {:>3.0f} | {}".format(x1, x2, "  ".join(cells)))

    xor = res["XOR"]
    print("\n" + "-" * 78)
    print("XOR ANALYSIS")
    print("-" * 78)
    errs = xor["model"].errors_per_epoch
    print("Misclassifications per epoch (first 20 epochs):")
    print("  " + "  ".join(str(e) for e in errs[:20]))
    period = _detect_period(errs)
    if period:
        print(f"Repeating pattern detected with period {period}: "
              f"{errs[-period:]}  -> the weight updates cycle forever.")
    wrong_rows = [i for i in range(4) if xor["pred"][i] != xor["y"][i]]
    print(f"Still misclassified after {xor['model'].epochs_run} epochs: "
          f"{[list(map(int, GATE_INPUTS[i])) for i in wrong_rows]}")
    print("""
WHY A SINGLE LAYER FAILS ON XOR
  XOR requires output 1 for (0,1) and (1,0), and output 0 for (0,0) and (1,1).
  A single-layer perceptron computes  y = step(w1*x1 + w2*x2 + b), so the
  positive class must lie entirely on one side of one straight line.
  Writing the four required inequalities:
        (0,0) -> 0 :            b  < 0
        (0,1) -> 1 :      w2  + b >= 0   =>  w2 >= -b > 0
        (1,0) -> 1 :      w1  + b >= 0   =>  w1 >= -b > 0
        (1,1) -> 0 : w1 + w2 + b  < 0   =>  w1 + w2 < -b
  The middle two give w1 + w2 >= -2b, and since -b > 0 that means
  w1 + w2 >= -2b > -b, which directly contradicts the fourth inequality.
  The system has no solution, so no line exists, the error count never reaches
  zero, and the learning rule keeps cycling through the same corrections.
  AND, OR and NAND each have a consistent solution, so they converge.
  Fix: use a hidden layer (an MLP) or a non-linear feature such as x1*x2.""")
    pause()


def _detect_period(seq, max_p=6):
    if len(seq) < 2 * max_p:
        return None
    tail = seq[-2 * max_p:]
    for p in range(1, max_p + 1):
        if all(tail[i] == tail[i - p] for i in range(p, len(tail))):
            return p
    return None


# --------------------------------------------------------------------------- #
# Option 7 - Perceptron Feature Removal Challenge
# --------------------------------------------------------------------------- #
def removal_results(D):
    base = Perceptron().fit(D["Xtr"], D["ytr"])
    base_acc = accuracy(D["yte"], base.predict(D["Xte"]))
    rows = []
    for f in range(4):
        keep = [i for i in range(4) if i != f]
        m = Perceptron().fit(D["Xtr"][:, keep], D["ytr"])
        acc = accuracy(D["yte"], m.predict(D["Xte"][:, keep]))
        rows.append({
            "removed": FEATURE_NAMES[f], "kept": [FEATURE_NAMES[i] for i in keep],
            "acc": acc, "drop": base_acc - acc, "epochs": m.epochs_run,
            "converged": m.converged,
        })
    rows.sort(key=lambda r: -r["drop"])
    return base, base_acc, rows


def option_7(D):
    banner("7. PERCEPTRON FEATURE REMOVAL CHALLENGE")
    base, base_acc, rows = removal_results(D)
    print(f"Baseline (all 4 features): accuracy {base_acc:.2f}%  "
          f"epochs {base.epochs_run}  converged "
          f"{'YES' if base.converged else 'NO'}")
    print(f"Baseline weights: {weight_string(base.w, base.b, FEATURE_NAMES)}")
    print("\n{:<12} {:<34} {:>10} {:>12} {:>8}".format(
        "Removed", "Remaining Features", "Accuracy", "Drop (pts)", "Epochs"))
    print("-" * 78)
    for r in rows:
        print("{:<12} {:<34} {:>9.2f}% {:>12.2f} {:>8}".format(
            r["removed"], ", ".join(r["kept"]), r["acc"], r["drop"], r["epochs"]))
    print("-" * 78)
    top = rows[0]
    print(f"MOST IMPORTANT FEATURE : {top['removed']} "
          f"(removing it costs {top['drop']:.2f} accuracy points)")
    print(f"LEAST IMPORTANT FEATURE: {rows[-1]['removed']} "
          f"(removing it costs {rows[-1]['drop']:.2f} points)")
    pause()


# --------------------------------------------------------------------------- #
# Option 8 - Perceptron Classification Report
# --------------------------------------------------------------------------- #
def option_8(D):
    banner("8. PERCEPTRON CLASSIFICATION REPORT")
    print(f"Dataset source : {D['source']}")
    print(f"Samples        : {len(D['y'])}  "
          f"(train {len(D['ytr'])} / test {len(D['yte'])})")
    print(f"Class balance  : Genuine={int(np.sum(D['y'] == 0))}  "
          f"Forged={int(np.sum(D['y'] == 1))}")
    print(f"Preprocessing  : min-max normalisation to [0,1]; classes encoded 0/1")
    print(f"Hyperparams    : lr={LEARNING_RATE}  max_epochs={MAX_EPOCHS}  "
          f"activation=step")

    base = Perceptron().fit(D["Xtr"], D["ytr"])
    base_pred = base.predict(D["Xte"])
    base_acc = accuracy(D["yte"], base_pred)
    _, _, _, _, top5, n_border = borderline_info(D)

    print("\n--- A. Comparative experiment table ---")
    print("{:<26} {:<24} {:>7} {:>9} {:>11}".format(
        "Experiment", "Features / Setting", "Epochs", "Accuracy", "Converged"))
    print("-" * 82)

    def row(exp, setting, ep, acc, conv):
        print("{:<26} {:<24} {:>7} {:>8.2f}% {:>11}".format(
            exp, setting[:24], ep, acc, "YES" if conv else "NO"))

    row("Full model", "All 4 features", base.epochs_run, base_acc, base.converged)

    for r in feature_pair_results(D):
        row("Feature pair", r["label"], r["epochs"], r["acc"], r["converged"])

    for r in order_results(D):
        row("Sample order", r["label"] + " order", r["epochs"], r["acc"],
            r["converged"])

    _, _, rem = removal_results(D)
    for r in rem:
        row("Feature removal", "without " + r["removed"], r["epochs"],
            r["acc"], r["converged"])

    for gate, r in gate_results().items():
        row("Logic gate", gate + " truth table", r["model"].epochs_run,
            r["acc"], r["converged"])

    print("\n--- B. Final model parameters ---")
    for n, w in zip(FEATURE_NAMES, base.w):
        print(f"   w({n:<9}) = {w:+.6f}")
    print(f"   bias        = {base.b:+.6f}")
    print("   Boundary: " + " ".join(
        f"({w:+.4f})*{n} +" for n, w in zip(FEATURE_NAMES, base.w)) +
        f" ({base.b:+.4f}) = 0")

    print("\n--- C. Classification quality (test set) ---")
    tp = int(np.sum((base_pred == 1) & (D["yte"] == 1)))
    tn = int(np.sum((base_pred == 0) & (D["yte"] == 0)))
    fp = int(np.sum((base_pred == 1) & (D["yte"] == 0)))
    fn = int(np.sum((base_pred == 0) & (D["yte"] == 1)))
    prec = tp / (tp + fp) * 100 if tp + fp else 0.0
    rec = tp / (tp + fn) * 100 if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    print("   Confusion matrix          Predicted Genuine  Predicted Forged")
    print(f"   Actual Genuine            {tn:>17} {fp:>17}")
    print(f"   Actual Forged             {fn:>17} {tp:>17}")
    print(f"   Accuracy {base_acc:.2f}%   Precision {prec:.2f}%   "
          f"Recall {rec:.2f}%   F1 {f1:.2f}%")
    print(f"   Borderline samples (distance < {BORDER_MARGIN}): {n_border}")
    print(f"   Five nearest to the boundary   : test rows "
          f"{[int(i) for i in top5]}")

    print("\n--- D. Findings ---")
    pairs = feature_pair_results(D)
    orders = order_results(D)
    accs = [o["acc"] for o in orders]
    print(f"   * Best feature pair      : {pairs[0]['label']} "
          f"({pairs[0]['acc']:.2f}%)")
    print(f"   * Worst feature pair     : {pairs[-1]['label']} "
          f"({pairs[-1]['acc']:.2f}%)")
    print(f"   * Most critical feature  : {rem[0]['removed']} "
          f"(-{rem[0]['drop']:.2f} points when removed)")
    spread = max(accs) - min(accs)
    verdict = ("boundaries differ only slightly" if spread < 2
               else "order has a strong effect on the final boundary")
    print(f"   * Sample-order accuracy  : {min(accs):.2f}% - {max(accs):.2f}% "
          f"(spread {spread:.2f} pts); {verdict}")
    print(f"   * AND / OR / NAND converge; XOR never does, confirming that a")
    print(f"     single linear boundary cannot solve non-linearly separable data")
    print(f"   * Overall convergence    : "
          f"{'reached' if base.converged else 'not reached within ' + str(MAX_EPOCHS) + ' epochs'}")
    pause()


# --------------------------------------------------------------------------- #
# Menu driver
# --------------------------------------------------------------------------- #
MENU = """
+----------------------------------------------------------------+
|   PERCEPTRON DECISION BOUNDARY & LINEAR SEPARABILITY CHALLENGE |
|              Banknote Authentication Dataset                   |
+----------------------------------------------------------------+
|  1. Banknote Classification using Perceptron                   |
|  2. Decision Boundary Explorer                                 |
|  3. Feature Pair Separability Challenge                        |
|  4. Training Sample Order Challenge                            |
|  5. Borderline Sample Detector                                 |
|  6. XOR Limitation Demonstrator                                |
|  7. Perceptron Feature Removal Challenge                       |
|  8. Perceptron Classification Report                           |
|  9. Exit                                                       |
+----------------------------------------------------------------+"""


def build_data(path=None):
    X_raw, y, source = load_dataset(path)
    X = normalize(X_raw)
    Xtr, Xte, ytr, yte = stratified_split(X, y)
    return {"X": X, "X_raw": X_raw, "y": y, "source": source,
            "Xtr": Xtr, "Xte": Xte, "ytr": ytr, "yte": yte}


def main():
    random.seed(SEED)
    np.random.seed(SEED)
    path = sys.argv[1] if len(sys.argv) > 1 else None
    D = build_data(path)

    print(f"\nDataset loaded from: {D['source']}")
    print(f"Shape: {D['X'].shape[0]} samples x {D['X'].shape[1]} features "
          f"| Genuine={int(np.sum(D['y'] == 0))} Forged={int(np.sum(D['y'] == 1))}")
    if D["source"].startswith("SYNTHETIC"):
        print("NOTE: place BankNote_Authentication.csv beside this script to use "
              "the real data.")

    actions = {"1": option_1, "2": option_2, "3": option_3, "4": option_4,
               "5": option_5, "6": option_6, "7": option_7, "8": option_8}
    while True:
        print(MENU)
        choice = input("Enter your choice (1-9): ").strip()
        if choice == "9":
            print("Exiting. Goodbye!")
            break
        if choice in actions:
            actions[choice](D)
        else:
            print("Invalid choice. Please enter a number between 1 and 9.")


if __name__ == "__main__":
    main()


Dataset loaded from: SYNTHETIC (dataset file not found)
Shape: 1372 samples x 4 features | Genuine=762 Forged=610
NOTE: place BankNote_Authentication.csv beside this script to use the real data.

+----------------------------------------------------------------+
|   PERCEPTRON DECISION BOUNDARY & LINEAR SEPARABILITY CHALLENGE |
|              Banknote Authentication Dataset                   |
+----------------------------------------------------------------+
|  1. Banknote Classification using Perceptron                   |
|  2. Decision Boundary Explorer                                 |
|  3. Feature Pair Separability Challenge                        |
|  4. Training Sample Order Challenge                            |
|  5. Borderline Sample Detector                                 |
|  6. XOR Limitation Demonstrator                                |
|  7. Perceptron Feature Removal Challenge                       |
|  8. Perceptron Classification Report                           |

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls "/content/drive/MyDrive"

'Colab Notebooks'


In [3]:
!ls "/content/drive/MyDrive/Colab Notebooks"

'Copy of Untitled0.ipynb'		      Exercise5.ipynb
 Exercise10.ipynb			      Exercise6.ipynb
 Exercise1Perceptron.ipynb		      Exercise7.ipynb
 Exercise2Backpropagation.ipynb		      Exercise8.ipynb
'Exercise3Feedforward Neural Network.ipynb'   Exercise9.ipynb
 Exercise4.ipynb


In [4]:
%cd /content
!git clone https://github.com/YOUR_USERNAME/Neural-Network-Implementations.git

/content
Cloning into 'Neural-Network-Implementations'...
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
%cd /content
!git clone https://github.com/akshara2701/Neural-Network-Implementations.git